# Session linearization

This notebook turns the filled environmental time series into the final linear ML dataset. Sessions are used only during construction: rows are grouped by separate room/source keys, split when a configurable time gap is exceeded, and compressed through overlapping lookback windows. The saved final dataset intentionally drops source, location, session, record, and timestamp metadata so every row stands alone.

In [1]:
from pathlib import Path
import sys

import pandas as pd

MAL_DIR = Path.cwd()
if MAL_DIR.name != "MAL":
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "MAL")

sys.path.insert(0, str(MAL_DIR))

from scripts.linearize_session_windows import build_linearized_windows

INPUT_PATH = MAL_DIR / "data" / "processed" / "unified_environment_respondent_focus_score_filled.csv"
OUTPUT_PATH = MAL_DIR / "data" / "processed" / "linearized_session_windows_30min.csv"
REPORT_PATH = MAL_DIR / "data" / "processed" / "linearized_session_windows_30min_report.csv"

FEATURE_COLUMNS = ["temperature", "humidity", "light", "noise", "co2"]
GROUP_COLUMNS = ["source", "location_id"]
TARGET_COLUMN = "focus_score"
WINDOW_MINUTES = 30
SESSION_GAP_MINUTES = 30
METADATA_COLUMNS = [
    "source", "location_id", "linear_session_id", "window_end", "window_start",
    "source_record_id", "source_session_id", "session_elapsed_minutes",
]

INPUT_PATH, OUTPUT_PATH, REPORT_PATH

(WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/unified_environment_respondent_focus_score_filled.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/linearized_session_windows_30min.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/linearized_session_windows_30min_report.csv'))

## Generate or load the final linearized dataset

In [2]:
if OUTPUT_PATH.exists() and REPORT_PATH.exists():
    linearized_df = pd.read_csv(OUTPUT_PATH, low_memory=False)
    session_report = pd.read_csv(REPORT_PATH, low_memory=False)
else:
    source_df = pd.read_csv(INPUT_PATH, low_memory=False)
    linearized_df, session_report = build_linearized_windows(
        source_df,
        feature_columns=FEATURE_COLUMNS,
        target_column=TARGET_COLUMN,
        group_columns=GROUP_COLUMNS,
        window_minutes=WINDOW_MINUTES,
        session_gap_minutes=SESSION_GAP_MINUTES,
    )
    linearized_df.to_csv(OUTPUT_PATH, index=False)
    session_report.to_csv(REPORT_PATH, index=False)

run_summary = pd.DataFrame([
    {
        "input_csv": str(INPUT_PATH),
        "output_csv": str(OUTPUT_PATH),
        "rows": len(linearized_df),
        "columns": linearized_df.shape[1],
        "sessions_used_during_build": len(session_report),
        "window_minutes": WINDOW_MINUTES,
        "session_gap_minutes": SESSION_GAP_MINUTES,
        "target_missing": int(linearized_df["focus_score"].isna().sum()),
        "target_min": int(linearized_df["focus_score"].min()),
        "target_max": int(linearized_df["focus_score"].max()),
    }
])
run_summary

,input_csv,output_csv,rows,columns,sessions_used_during_build,window_minutes,session_gap_minutes,target_missing,target_min,target_max
0,c:\Users\piotr\Desktop\University\Semester4\SE...,c:\Users\piotr\Desktop\University\Semester4\SE...,840826,36,281,30,30,0,1,5


## Session checks kept in the separate report

In [3]:
session_report[["rows", "duration_minutes"]].describe()

,rows,duration_minutes
count,281.000000,281.000000
mean,2992.263345,7706.755931
std,4757.078008,23208.515458
min,1.000000,0.000000
25%,5.000000,28.666667
50%,107.000000,244.000000
75%,5461.000000,5461.000000
max,45843.000000,240030.916667


In [4]:
session_report.sort_values("duration_minutes", ascending=False).head(10)

,linear_session_id,source,location_id,rows,first_timestamp,last_timestamp,duration_minutes
23,homecoach_5min_2025__homecoach_5min_2025_unkno...,homecoach_5min_2025,homecoach_5min_2025_unknown_location,45843,2025-07-18T07:28:36Z,2025-12-31T23:59:31Z,240030.916667
15,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,25171,2024-04-11T10:48:03Z,2024-07-12T10:03:34Z,132435.516667
10,homecoach_5min_2023__homecoach_5min_2023_unkno...,homecoach_5min_2023,homecoach_5min_2023_unknown_location,24683,2023-08-20T07:53:47Z,2023-11-17T09:44:16Z,128270.483333
20,homecoach_5min_2025__homecoach_5min_2025_unkno...,homecoach_5min_2025,homecoach_5min_2025_unknown_location,24369,2025-01-01T00:02:30Z,2025-03-30T01:57:59Z,126835.483333
21,homecoach_5min_2025__homecoach_5min_2025_unkno...,homecoach_5min_2025,homecoach_5min_2025_unknown_location,20823,2025-03-30T03:02:59Z,2025-06-14T03:13:24Z,109450.416667
19,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,16113,2024-11-03T18:09:12Z,2024-12-31T23:54:31Z,83865.316667
12,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,12646,2024-01-01T00:02:30Z,2024-02-15T17:52:45Z,65870.250000
17,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,12352,2024-08-22T18:13:48Z,2024-10-06T07:44:03Z,64170.250000
11,homecoach_5min_2023__homecoach_5min_2023_unkno...,homecoach_5min_2023,homecoach_5min_2023_unknown_location,12388,2023-11-17T14:34:16Z,2023-12-31T23:59:31Z,63925.250000
13,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,12083,2024-02-16T07:47:45Z,2024-03-31T01:58:00Z,63010.250000


## Final dataset checks

In [5]:
metadata_check = pd.DataFrame([
    {"column": column, "present_in_final_dataset": column in linearized_df.columns}
    for column in METADATA_COLUMNS
])
metadata_check

,column,present_in_final_dataset
0,source,False
1,location_id,False
2,linear_session_id,False
3,window_end,False
4,window_start,False
5,source_record_id,False
6,source_session_id,False
7,session_elapsed_minutes,False


In [6]:
target_distribution = (
    linearized_df["focus_score"]
    .value_counts()
    .sort_index()
    .rename_axis("focus_score")
    .reset_index(name="rows")
)
target_distribution

,focus_score,rows
0,1,9429
1,2,89550
2,3,412168
3,4,234725
4,5,94954


In [7]:
feature_coverage = pd.DataFrame([
    {
        "feature": feature,
        "latest_non_null": int(linearized_df[f"{feature}_latest"].notna().sum()),
        "window_count_nonzero": int((linearized_df[f"{feature}_count"] > 0).sum()),
        "mean_non_null": int(linearized_df[f"{feature}_mean"].notna().sum()),
    }
    for feature in FEATURE_COLUMNS
])
feature_coverage

,feature,latest_non_null,window_count_nonzero,mean_non_null
0,temperature,840826,840826,840826
1,humidity,840826,840826,840826
2,light,840826,840826,840826
3,noise,840826,840826,840826
4,co2,840826,840826,840826


In [8]:
assert linearized_df["focus_score"].notna().all()
assert linearized_df["focus_score"].between(1, 5).all()
assert not any(column in linearized_df.columns for column in METADATA_COLUMNS)
print("Final dataset is saved with only target + engineered feature columns.")

Final dataset is saved with only target + engineered feature columns.


## Preview of the holy grail dataset

In [9]:
linearized_df.head(20)

,focus_score,temperature_latest,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_count,temperature_range,humidity_latest,humidity_mean,...,noise_std,noise_count,noise_range,co2_latest,co2_mean,co2_min,co2_max,co2_std,co2_count,co2_range
0,2,17.7,17.700000,17.7,17.7,0.000000,1,0.0,56.0,56.000000,...,0.000000,1,0.0,2305.0,2305.000000,2305.0,2305.0,0.000000,1,0.0
1,2,17.9,17.800000,17.7,17.9,0.141421,2,0.2,56.0,56.000000,...,4.242641,2,6.0,2250.0,2277.500000,2250.0,2305.0,38.890873,2,55.0
2,3,18.3,17.966667,17.7,18.3,0.305505,3,0.6,57.0,56.333333,...,5.033223,3,10.0,2286.0,2280.333333,2250.0,2305.0,27.934447,3,55.0
3,3,18.9,18.200000,17.7,18.9,0.529150,4,1.2,57.0,56.500000,...,8.261356,4,19.0,2266.0,2276.750000,2250.0,2305.0,23.907809,4,55.0
4,3,19.1,18.380000,17.7,19.1,0.609918,5,1.4,57.0,56.600000,...,9.731393,5,23.0,2238.0,2269.000000,2238.0,2305.0,27.000000,5,67.0
5,3,19.3,18.533333,17.7,19.3,0.662319,6,1.6,57.0,56.666667,...,9.416298,6,23.0,2255.0,2266.666667,2238.0,2305.0,24.816661,6,67.0
6,3,19.5,18.833333,17.9,19.5,0.615359,6,1.6,56.0,56.666667,...,10.315038,6,24.0,2234.0,2254.833333,2234.0,2286.0,19.166812,6,52.0
7,3,19.6,19.116667,18.3,19.6,0.475044,6,1.3,56.0,56.666667,...,8.795832,6,24.0,2212.0,2248.500000,2212.0,2286.0,26.105555,6,74.0
8,2,19.8,19.366667,18.9,19.8,0.332666,6,0.9,56.0,56.500000,...,2.926887,6,7.0,2133.0,2223.000000,2133.0,2266.0,47.833043,6,133.0
9,4,19.8,19.516667,19.1,19.8,0.278687,6,0.7,55.0,56.166667,...,2.898275,6,7.0,1849.0,2153.500000,1849.0,2255.0,155.232406,6,406.0
